# Análise de Arboviroses no RN: Impactos Climáticos e Socioambientais na Dengue
---

## 1. Introdução e Escopo do Projeto
Este notebook implementa o pipeline de Ciência de Dados para analisar a incidência de **Dengue** no estado do Rio Grande do Norte (RN). O objetivo principal é entender como variáveis climáticas (temperatura e precipitação) e de infraestrutura (saneamento básico) correlacionam-se aos surtos epidemiológicos.

### Metodologia de Amostragem
Para garantir a representatividade geográfica, populacional e socioeconômica do estado sem gerar redundâncias, selecionamos os **dois municípios mais populosos de cada uma das 4 mesorregiões do RN**.

> **Nota de Validação:** Municípios da Região Metropolitana de Natal (como Parnamirim) foram intencionalmente excluídos, dado que seu comportamento epidemiológico e socioambiental apresenta forte colinearidade e similaridade com os dados da capital, que já está inclusa na pesquisa.

---

## Desenho Amostral dos Municípios (Por Mesorregião)

| Mesorregião | Município | Código IBGE | População Estimada | Perfil Estratégico |
| :--- | :--- | :--- | :--- | :--- |
| **Oeste Potiguar** | Mossoró | `2408003` | 264.577 | Polo urbano do semiárido, calor extremo. |
| | Pau dos Ferros | `2409407` | 30.479 | Polo do extremo oeste, transição geográfica. |
| **Leste Potiguar** | Natal | `2408102` | 751.300 | Capital, alta densidade, clima litorâneo úmido. |
| | Touros | `2414407` | 33.035 | Litoral norte, transição turismo/área rural. |
| **Central Potiguar** | Caicó | `2402006` | 61.146 | Polo do Seridó, semiárido extremo, armazenamento de água. |
| | Macau | `2407203` | 27.369 | Litoral salino, transição árida, desafios de esgoto. |
| **Agreste Potiguar** | Santa Cruz | `2411205` | 37.313 | Polo do Trairi, relevo acidentado, semiárido. |
| | João Câmara | `2405801` | 33.290 | Polo do Mato Grande, transição de ventos e seca. |

## 2. Configuração do Ambiente e Importação de Bibliotecas
Nesta etapa, preparamos o ambiente carregando os pacotes necessários para manipulação de dados, requisições de API, análises estatísticas e visualização gráfica.

*   **Bibliotecas Padrão do Python:** `gc`, `io`, `os`, `sys`, `time`, `warnings`, `datetime`, `pathlib.Path`
*   **Manipulação e Análise de Dados:** `numpy`, `pandas`
*   **Coleta de Dados:** `requests`, `zipfile`, `openmeteo-requests`, `requests_cache`, `retry_requests`
*   **Visualização:** `matplotlib.pyplot`, `matplotlib.dates`, `matplotlib.ticker`, `seaborn`, `matplotlib.patches`
*   **Estatística e Utilidades:** `epiweeks`, `scipy`, `scipy.stats`, `statsmodels`
*   **Google Colab:** `google.colab.drive`

### 2.1 Imports e Configurações Essenciais

In [1]:
import subprocess, sys

DEPS = [
    ('pandas',            'pandas'),
    ('numpy',             'numpy'),
    ('requests',          'requests'),
    ('matplotlib',        'matplotlib'),
    ('seaborn',           'seaborn'),
    ('pyarrow',           'pyarrow'),
    ('epiweeks',          'epiweeks'),
    ('scipy',             'scipy'),
    ('statsmodels',       'statsmodels'),
    ('openmeteo-requests','openmeteo_requests'),
    ('requests-cache',    'requests_cache'),
    ('retry-requests',    'retry_requests'),
]

print(' Verificando dependências...')
for pkg, imp_name in DEPS:
    try:
        __import__(imp_name)
        print(f'  ✔ {pkg}')
    except ImportError:
        print(f'  Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  ✔ {pkg} instalado')

print('\n ✅ Todas as dependências prontas!')

 Verificando dependências...
  ✔ pandas
  ✔ numpy
  ✔ requests
  ✔ matplotlib
  ✔ seaborn
  ✔ pyarrow
  Instalando epiweeks...
  ✔ epiweeks instalado
  ✔ scipy
  ✔ statsmodels
  Instalando openmeteo-requests...
  ✔ openmeteo-requests instalado
  Instalando requests-cache...
  ✔ requests-cache instalado
  Instalando retry-requests...
  ✔ retry-requests instalado

 ✅ Todas as dependências prontas!


In [2]:
# 1. Bibliotecas padrão do Python
import gc
import io
import os
import sys
import time
import warnings
from datetime import datetime, date
from pathlib import Path

# 2. Manipulação e análise de dados
import numpy as np
import pandas as pd

# 3. Coleta de dados
import requests
import zipfile
import openmeteo_requests
import requests_cache
from retry_requests import retry

# 4. Visualização de dados
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch as _Patch

# 5. Estatística, datas epidemiológicas e outros utilitários
import datetime as _dt
from epiweeks import Week
from scipy import stats
from scipy.stats import spearmanr

# 6. Statsmodels (usado nas análises complementares)
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.tsa.seasonal import seasonal_decompose

# 7. Google Colab (para montar o drive)
from google.colab import drive

print('✅ Imports configurados')

✅ Imports configurados


### 2.2 Constantes do Ambiente
Define os municípios, doenças, período de análise e URLs das fontes de dados.

> **Notas sobre os identificadores:**
> - **IBGE 6 dígitos** - código encurtado (oficial tem 7 dígitos) usado nas bases

In [3]:
# -- Municípios --
MUNICIPIOS = {
    # Oeste potiguar
    '240940': 'PAU DOS FERROS', # 2409407
    '240800': 'MOSSORO',        # 2408003
    # Leste litorâneo
    '240810': 'NATAL',          # 2408102
    '241440': 'TOUROS',         # 2414407
    # Central potiguar
    '240200': 'CAICO',          # 2402006
    '240720': 'MACAU',          # 2407203
    # Agreste
    '241120': 'SANTA CRUZ',     # 2411205
    '240580': 'JOAO CAMARA'     # 2405801
}

# -- Período de análise --
ANOS        = list(range(2003, 2023))
DATA_INICIO = '2003-01-01'
DATA_FIM    = '2022-12-31'

# -- URLs --
# Arboviroses
URL_DENGUE = 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR{ano_str}.csv.zip'

# Dados climáticos
URL_OPENMETEO_HIST = 'https://archive-api.open-meteo.com/v1/archive'

# Dados de saneamento
# TODO

# -- Paleta de cores --
# TODO
CORES_CIDADE =  {

}

# -- Padronização --
MES_NOME = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
            7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}

MUNICIPIOS_COLS = ['COMUNINF', 'ID_MN_RESI']

print('Constantes definidas')
print(f'   Municípios  : {list(MUNICIPIOS.values())}')
print(f'   IBGE-6      : {MUNICIPIOS}')
print(f'   Período     : {DATA_INICIO} → {DATA_FIM}')


Constantes definidas
   Municípios  : ['PAU DOS FERROS', 'MOSSORO', 'NATAL', 'TOUROS', 'CAICO', 'MACAU', 'SANTA CRUZ', 'JOAO CAMARA']
   IBGE-6      : {'240940': 'PAU DOS FERROS', '240800': 'MOSSORO', '240810': 'NATAL', '241440': 'TOUROS', '240200': 'CAICO', '240720': 'MACAU', '241120': 'SANTA CRUZ', '240580': 'JOAO CAMARA'}
   Período     : 2003-01-01 → 2022-12-31


## 2.3 Diretórios e configuração de armazenamento

**Estratégia de persistência:**
- Se o Google Drive estiver montado (Colab), os dados são salvos em `MyDrive/eda-arboviroses/`.
- Caso contrário, usa pasta local `./local_data/`.

A estrutura de diretórios criada será:

```
eda-arboviroses/
├── parquets/
│   ├── clima/
│   │   └── clima03-22.parquet
│   ├── dengue/
│   │   ├── dengue03.parquet
│   │   ├── ...
│   │   └── dengue22.parquet
│   ├── saneamento/
│   │   ├── ?
│   │   ├── ...
│   │   └── ?
│   └── sinan.parquet
└── visualizations
    └── [imagem].png
```

In [4]:
try:
    drive.mount('/content/drive')
    DRIVE_BASE = Path('/content/drive/MyDrive/eda-arboviroses')
    print('Google Drive montado')
except Exception:
    DRIVE_BASE = Path('./local_data')
    print('Modo local — usando ./local_data/')

# Cria subdiretórios
PARQUET_DIR = DRIVE_BASE / 'parquets'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
(PARQUET_DIR / 'clima').mkdir(exist_ok=True)
(PARQUET_DIR / 'dengue').mkdir(exist_ok=True)
(PARQUET_DIR / 'saneamento').mkdir(exist_ok=True)

VISUALIZATION_DIR = DRIVE_BASE / 'visualizations'
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

# Paths dos parquets consolidados (TODO)
PQ_SINAN = PARQUET_DIR / 'sinan.parquet'

print(f' Base               : {DRIVE_BASE}')
print(f' ├── Parquets       : {PARQUET_DIR}')
print(f' │   ├── SINAN      : {PQ_SINAN}')
print(f' └── Visualizations : {VISUALIZATION_DIR}')

Modo local — usando ./local_data/
 Base               : local_data
 ├── Parquets       : local_data/parquets
 │   ├── SINAN      : local_data/parquets/sinan.parquet
 └── Visualizations : local_data/visualizations


## 3. Pipeline de Coleta de Dados (Data Ingestion)
O projeto consome dados de três fontes distintas para construir a matriz de análise:

### Fontes de Dados:
1.  **Epidemiológicos (Dengue):** Dados extraídos do **DATASUS (SINAN)**. Foco no volume de casos notificados por semana/ano epidemiológico por município.
2.  **Saneamento Básico:** Dados extraídos do **SNIS (Sistema Nacional de Informações sobre Saneamento)**. Janela temporal utilizada: **2003 a 2022** (período consolidado sem a transição para o novo formato SNISA). Foco em indicadores de atendimento de água, coleta de esgoto e perdas na rede.
3.  **Clima:** Histórico de precipitação (chuva) e temperatura consumido via API do **Open-Meteo**, utilizando as coordenadas geográficas dos municípios polo.

*Abaixo, estruturamos as funções de carga de arquivos locais e requisições externas para os códigos IBGE selecionados.*

### 3.1 Extração de Dados SINAN (Dengue)

A função `carregar_ou_processar_sinan` é responsável por obter os dados de casos de Dengue do sistema SINAN (DATASUS). Ela implementa uma estratégia de _cache-first_ para otimizar a performance:

1.  **Verificação de Cache:** Primeiramente, tenta carregar os dados de um arquivo Parquet local, se ele existir e não estiver corrompido. Isso evita downloads e reprocessamento desnecessários.
2.  **Download:** Se o cache não estiver disponível, o arquivo compactado `.csv.zip` é baixado diretamente do S3 do DATASUS.
3.  **Processamento em Chunks:** O arquivo CSV é lido em blocos (_chunks_) para lidar eficientemente com grandes volumes de dados sem sobrecarregar a memória. Cada chunk é processado para:
    *   **Filtragem de Municípios:** Identificação e seleção apenas dos municípios de interesse definidos em `MUNICIPIOS`.
    *   **Detecção de Colunas:** Auto-detecção das colunas de código de município e de semana epidemiológica (`SEM_PRI`).
    *   **Padronização de Semana Epidemiológica:** A coluna `SEM_PRI` é tratada para dois formatos históricos:
        *   **WWYYYY (até 2006):** Formato como '242006' (semana 24, ano 2006).
        *   **YYYYWW (a partir de 2007):** Formato como '200724' (ano 2007, semana 24).
        Ambos os formatos são padronizados para `YYYYWW` e validados quanto ao ano e número da semana (1-53), com tratamento robusto para valores inválidos.
    *   **Agregação:** Os dados são agregados por `MUNICIPIO` e `SEMANA_EPI`, somando os casos (`CASOS`) e adicionando a coluna `DOENCA` com o valor 'Dengue'.
4.  **Persistência:** Após o processamento de todos os chunks, os dados agregados são concatenados e salvos em um novo arquivo Parquet local, pronto para futuras análises.

In [5]:
def carregar_ou_processar_sinan(url_base, ano_referencia, caminho_parquet):
    """
    Função principal para carregar ou processar dados do SINAN para um dado ano.
    Prioriza o cache local (arquivo Parquet) para otimizar o desempenho.
    Caso contrário, baixa o arquivo CSV.zip do S3, filtra os dados por município
    e semana epidemiológica, agrega os casos e persiste o resultado em Parquet.
    """
    # 1. Tenta carregar do cache (arquivo Parquet)
    if caminho_parquet.exists():
        try:
            df_armazenado = pd.read_parquet(caminho_parquet)
            if not df_armazenado.empty:
                print(f'   Cache: {caminho_parquet.name}')
                return None # Retorna None para indicar que o cache foi usado
        except Exception as e:
            print(f'    Cache corrompido ({e}) → re-baixando')
            caminho_parquet.unlink(missing_ok=True) # Exclui o cache corrompido

    # 2. Download do arquivo ZIP
    url = url_base.format(ano_str=str(ano_referencia)[-2:])
    print(f'   Baixando {ano_referencia}…')
    try:
        resposta = requests.get(url, timeout=180)
    except Exception as e:
        print(f'   Erro de conexão: {e}')
        return None
    if resposta.status_code != 200 or len(resposta.content) < 1000:
        print(f'   HTTP {resposta.status_code} ou resposta vazia')
        return None

    # 3. Leitura do CSV em blocos (chunks)
    lista_registros = []
    try:
        with zipfile.ZipFile(io.BytesIO(resposta.content)) as zf:
            arquivos_csv = [nome for nome in zf.namelist() if nome.lower().endswith('.csv')]
            if not arquivos_csv:
                print('   Nenhum arquivo CSV dentro do ZIP')
                return None
            for nome_arquivo in arquivos_csv:
                for bloco_dados in pd.read_csv(
                    zf.open(nome_arquivo),
                    sep=',',
                    dtype=str,
                    encoding='latin1',
                    chunksize=50_000,
                    low_memory=False
                ):
                    # Detecta a melhor coluna de município presente no bloco de dados
                    coluna_municipio = None
                    for col in MUNICIPIOS_COLS:
                        if col in bloco_dados.columns:
                            coluna_municipio = col
                            break
                    if not coluna_municipio: continue

                    # Filtra pelos municípios de interesse e mapeia para nomes amigáveis
                    bloco_dados = bloco_dados[bloco_dados[coluna_municipio].astype(str).str[:6].isin(MUNICIPIOS.keys())].copy()
                    if bloco_dados.empty: continue
                    bloco_dados['MUNICIPIO'] = bloco_dados[coluna_municipio].astype(str).str[:6].map(MUNICIPIOS)

                    # Detecta coluna de semana epidemiológica ('SEM_PRI')
                    if 'SEM_PRI' not in bloco_dados.columns: continue
                    sem_pri_bruta = bloco_dados['SEM_PRI'].astype(str).str.strip()

                    if ano_referencia <= 2006:
                        # Formato WWYYYY, ex: '242006' -> ano '2006', semana '24'
                        # Primeiro, filtra strings com 6 dígitos
                        mascara_tamanho = sem_pri_bruta.str.match(r'^\d{6}$', na=False)
                        sem_pri_bruta_temp = sem_pri_bruta[mascara_tamanho]
                        bloco_dados_temp = bloco_dados[mascara_tamanho].copy()

                        if bloco_dados_temp.empty:
                            bloco_dados = pd.DataFrame() # Garante que o bloco_dados esteja vazio
                            continue

                        ano_str = sem_pri_bruta_temp.str[2:6]
                        semana_str = sem_pri_bruta_temp.str[:2]

                        # Usa pd.to_numeric com errors='coerce' para robustez
                        valor_ano = pd.to_numeric(ano_str, errors='coerce')
                        valor_semana = pd.to_numeric(semana_str, errors='coerce')

                        # Valida valores de ano e semana, filtrando NaNs
                        mascara_valores_validos = (
                            ~valor_ano.isna() & valor_ano.isin(ANOS) &
                            ~valor_semana.isna() & valor_semana.between(1, 53)
                        )

                        bloco_dados = bloco_dados_temp[mascara_valores_validos].copy() # Filtro final para este ramo
                        if not bloco_dados.empty:
                            bloco_dados['SEMANA_EPI'] = (ano_str[mascara_valores_validos] + semana_str[mascara_valores_validos])

                    else:
                        # Formato YYYYWW, ex: '200724' -> ano '2007', semana '24'
                        bloco_dados['SEMANA_EPI'] = sem_pri_bruta
                        bloco_dados = bloco_dados.dropna(subset=['SEMANA_EPI']).copy() # Adiciona copy aqui

                        if bloco_dados.empty: continue

                        # Primeiro, filtra para garantir que é uma string de 6 dígitos
                        mascara_tamanho = bloco_dados['SEMANA_EPI'].str.match(r'^\d{6}$', na=False)
                        bloco_dados = bloco_dados[mascara_tamanho].copy()

                        if bloco_dados.empty: continue

                        valor_ano = pd.to_numeric(bloco_dados['SEMANA_EPI'].str[:4], errors='coerce')
                        valor_semana = pd.to_numeric(bloco_dados['SEMANA_EPI'].str[4:], errors='coerce')

                        # Valida consistência de ano e semana após coercão
                        mascara_valores_validos = (
                            ~valor_ano.isna() & valor_ano.isin(ANOS) &
                            ~valor_semana.isna() & valor_semana.between(1, 53)
                        )
                        bloco_dados = bloco_dados[mascara_valores_validos].copy() # Adiciona copy aqui

                    if bloco_dados.empty: continue

                    # Agrega casos por município e semana epidemiológica, adiciona ANO
                    df_agregado = (bloco_dados.groupby(['MUNICIPIO','SEMANA_EPI'])
                              .size().reset_index(name='CASOS'))
                    df_agregado['ANO'] = df_agregado['SEMANA_EPI'].str[:4].astype(int)

                    # Adiciona o bloco agregado à lista de registros
                    lista_registros.append(df_agregado)
                    del bloco_dados, df_agregado
                    gc.collect()

    except zipfile.BadZipFile as e:
        print(f'   ZIP inválido: {e}')
        return None
    except Exception as e:
        print(f'   Erro na leitura: {e}')
        return None

    if not lista_registros:
        print(f'   Nenhum registro válido em {ano_referencia}')
        return None

    df_final = pd.concat(lista_registros, ignore_index=True)
    df_final.to_parquet(caminho_parquet, index=False)
    print(f'   {caminho_parquet.name} ({len(df_final):,} semanas · {df_final["CASOS"].sum():,} casos)')
    del df_final, lista_registros; gc.collect()

In [6]:
# -- Loop principal SINAN --
print(f'🦟 COLETA SINAN - {ANOS[0]}-{ANOS[-1]}')
print('=' * 60)

for ano in ANOS:
    p = PARQUET_DIR / 'dengue' / f'dengue{str(ano)[-2:]}.parquet'
    carregar_ou_processar_sinan(URL_DENGUE, ano, p)
    time.sleep(0.15)

# Consolida todos os parquets parciais
pqs = sorted((PARQUET_DIR/ 'dengue').glob('dengue*.parquet'))
if pqs:
    df_sinan = pd.concat([pd.read_parquet(p) for p in pqs], ignore_index=True)
    df_sinan.to_parquet(PQ_SINAN, index=False)
    print(f'\n SINAN consolidado → {PQ_SINAN.name}')
    print(f'   Shape     : {df_sinan.shape}')
    print(f'   Período   : {df_sinan["SEMANA_EPI"].min()} → {df_sinan["SEMANA_EPI"].max()}')
    print(f'   Municípios: {sorted(df_sinan["MUNICIPIO"].unique())}')
    print(f'   Casos     : {df_sinan["CASOS"].sum():,}')
    display(df_sinan.groupby(["MUNICIPIO", "ANO"])["CASOS"].sum().unstack(fill_value=0))
else:
    print(' Nenhum parquet SINAN encontrado. Verifique a conexão.')
    df_sinan = pd.DataFrame()

🦟 COLETA SINAN - 2003-2022
   Baixando 2003…
   dengue03.parquet (263 semanas · 11,955 casos)
   Baixando 2004…
   dengue04.parquet (154 semanas · 1,311 casos)
   Baixando 2005…
   dengue05.parquet (268 semanas · 3,222 casos)
   Baixando 2006…
   dengue06.parquet (291 semanas · 4,833 casos)
   Baixando 2007…
   dengue07.parquet (376 semanas · 2,418 casos)
   Baixando 2008…
   dengue08.parquet (585 semanas · 5,007 casos)
   Baixando 2009…
   dengue09.parquet (101 semanas · 196 casos)
   Baixando 2010…
   dengue10.parquet (243 semanas · 1,580 casos)
   Baixando 2011…
   dengue11.parquet (1,739 semanas · 5,466 casos)
   Baixando 2012…
   dengue12.parquet (467 semanas · 4,220 casos)
   Baixando 2013…
   dengue13.parquet (259 semanas · 2,192 casos)
   Baixando 2014…
   dengue14.parquet (226 semanas · 1,143 casos)
   Baixando 2015…
   dengue15.parquet (226 semanas · 1,814 casos)
   Baixando 2016…
   dengue16.parquet (161 semanas · 3,356 casos)
   Baixando 2017…
   dengue17.parquet (101 seman

ANO,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
MUNICIPIO,,,,,,,,,,,,,,,,,,,,
CAICO,141,325,639,169,71,39,2,57,174,285,724,540,132,59,3,175,323,2,5,38
JOAO CAMARA,125,29,13,87,52,217,8,9,18,2,0,24,17,97,6,150,89,1,5,85
MACAU,37,3,70,93,104,195,1,0,76,42,2,1,68,36,1,1,2,1,12,36
MOSSORO,191,21,162,198,241,1411,127,158,1475,1207,114,110,1155,2666,1109,580,48,122,93,213
NATAL,10853,846,1498,4082,1568,1850,66,1054,3567,2499,506,423,450,446,68,881,1402,270,267,2601
PAU DOS FERROS,326,78,619,68,138,172,0,297,74,117,385,37,5,18,0,2,15,0,0,1
SANTA CRUZ,253,9,214,110,241,1076,5,23,97,32,436,6,3,3,1,5,10,2,4,12
TOUROS,59,1,0,18,17,25,0,2,4,14,3,1,12,1,1,38,14,1,0,5


In [7]:
df_sinan.head()

,MUNICIPIO,SEMANA_EPI,CASOS,ANO
0,MOSSORO,200330,1,2003
1,CAICO,200303,2,2003
2,CAICO,200304,1,2003
3,CAICO,200305,3,2003
4,CAICO,200306,3,2003


3.2 Extração de Dados Open-Meteo

In [ ]:
#TODO

3.3 Extração de Dados de Saneamento


In [ ]:
#TODO